In [1]:
#!unzip drive/MyDrive/Trajectory_Anomaly/archive.zip

In [2]:
import torch
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
import pandas as pd
import numpy as np

def preprocessing(df):

    df['datetime'] = pd.to_datetime(df['datetime'] + " " + df['time'])

    # Shift the arrays to line up the past and present on the same row
    df['prev_lat'] = df['latitude'].shift(1)
    df['prev_lon'] = df['longitude'].shift(1)
    df['prev_time'] = df['datetime'].shift(1)


    df['delta_time'] = (df['datetime'] - df['prev_time']).dt.total_seconds().fillna(0)

    # Distance (Vectorized Haversine)
    lat1 = np.radians(df['prev_lat'])
    lon1 = np.radians(df['prev_lon'])
    lat2 = np.radians(df['latitude'])
    lon2 = np.radians(df['longitude'])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    R = 6371000 # Earth's radius in meters
    df['distance'] = (R * c).fillna(0)


    df['speed'] = np.where(df['delta_time'] != 0, df['distance'] / df['delta_time'], 0)


    df['prev_speed'] = df['speed'].shift(1).fillna(0)
    df['acc'] = np.where(df['delta_time'] != 0, (df['speed'] - df['prev_speed']) / df['delta_time'], 0)


    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - (np.sin(lat1) * np.cos(lat2) * np.cos(dlon))
    theta = np.degrees(np.arctan2(x, y))
    df['bearing'] = (theta + 360) % 360
    df['bearing'] = df['bearing'].fillna(0)


    df['prev_bearing'] = df['bearing'].shift(1).fillna(0)
    diff = np.abs(df['bearing'] - df['prev_bearing'])
    df['change_in_direction'] = np.minimum(diff, 360 - diff)


    is_slow = df['speed'] <= 0.3


    blocks = (~is_slow).cumsum()


    cumulative_time = df.groupby(blocks)['delta_time'].cumsum() * is_slow


    df['is_stationary'] = (cumulative_time >= 60).astype(int)


    df = df.drop(
        columns=[
            "zero", "altitude", "date", "time", "datetime",
            "prev_lat", "prev_lon", "prev_time", "prev_speed", "prev_bearing", "bearing"
        ],
        errors='ignore'
    )


    df = df.iloc[1:].reset_index(drop=True)
    df = df[['latitude', 'longitude', 'distance', 'speed', 'change_in_direction', 'acc', 'delta_time', 'is_stationary']]

    return df

In [4]:
import os
root = os.getcwd()
Data = os.path.join(root, "Geolife Trajectories 1.3/Data")
dir = os.listdir(Data)

In [5]:
# test_dir = os.path.join(Data, dir[0])
# trajectory_dir = os.path.join(test_dir, "Trajectory")
# main_data_dir = os.listdir(trajectory_dir)
# df = pd.read_csv(os.path.join(trajectory_dir, main_data_dir[0]), header=None, skiprows = 6)
# df.columns = [
#   "latitude",
#   "longitude",
#   "zero",
#   "altitude",
#   "date",
#   "datetime",
#   "time"
# ]
# df


In [6]:
#Fitting StandardScaler on every users instance
import gc
from sklearn.preprocessing import StandardScaler

num_cols = ['speed', 'acc', 'change_in_direction', 'is_stationary']
scaler = StandardScaler()
all_features = []
for i in range(40, len(dir)):
  test_dir = os.path.join(Data, dir[i])
  print(f"Testing in Directory User: {test_dir}")
  trajectory_dir = os.path.join(test_dir, "Trajectory")
  main_data_dir = os.listdir(trajectory_dir)
  for j in range(len(main_data_dir)):
    df = pd.read_csv(os.path.join(trajectory_dir, main_data_dir[j]), header=None, skiprows = 6)
    df.columns = [
      "latitude",
      "longitude",
      "zero",
      "altitude",
      "date",
      "datetime",
      "time"
    ]

    df = preprocessing(df)
    features_array = df[num_cols].to_numpy(dtype=np.float32)
    all_features.append(features_array)
master_array = np.vstack(all_features)
scaler.fit(master_array)
import joblib
joblib.dump(scaler, "scaler.pkl")

del master_array
del all_features
gc.collect()
torch.cuda.empty_cache()

In [7]:
import joblib
scaler = joblib.load("scaler.pkl")

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

class TrajectoryDataset(Dataset):
  def __init__(self, X):
    self.data = X

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    return torch.tensor(self.data[idx], dtype = torch.float32)

# Train - 1

In [9]:
#train - 1

import numpy as np
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

num_cols = ['speed', 'acc', 'change_in_direction', 'is_stationary']
sequence_length = 30

total = []

for i in range(40, 110):

    test_dir = os.path.join(Data, dir[i])
    print(f"Testing in Directory User: {test_dir}")

    trajectory_dir = os.path.join(test_dir, "Trajectory")
    main_data_dir = os.listdir(trajectory_dir)

    for j in range(len(main_data_dir)):

        file_path = os.path.join(
            trajectory_dir,
            main_data_dir[j]
        )

        # Read file
        df = pd.read_csv(
            file_path,
            header=None,
            skiprows=6
        )

        df.columns = [
            "latitude",
            "longitude",
            "zero",
            "altitude",
            "date",
            "datetime",
            "time"
        ]


        df = preprocessing(df)

        if len(df) <= sequence_length:
            continue

        df[num_cols] = scaler.transform(df[num_cols].to_numpy())
        data = df[num_cols].to_numpy(dtype=np.float32)


        n_sequences = len(data) - sequence_length


        sequences = np.lib.stride_tricks.sliding_window_view(
            data,
            window_shape=sequence_length,
            axis=0
        )


        sequences = sequences.transpose(0, 2, 1)

        total.append(sequences)

# Combine all files once
total = np.concatenate(total, axis=0)

dataset = TrajectoryDataset(total)

train_loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True
)

Testing in Directory User: /content/Geolife Trajectories 1.3/Data/147
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/007
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/054
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/123
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/058
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/158
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/045
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/160
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/125
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/094
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/178
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/137
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/046
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/099
Testing in Directory

/tmp/ipykernel_4624/3076979818.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['datetime'] + " " + df['time'])


Testing in Directory User: /content/Geolife Trajectories 1.3/Data/050
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/181
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/159
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/043
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/027
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/066
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/085
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/029
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/141
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/122
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/098
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/020
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/164
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/048
Testing in Directory

In [10]:
import torch.nn as nn

class GRU_AutoEncoder(nn.Module):
  def __init__(self, input_size):
    super().__init__()
    self.encoder = nn.GRU(
        input_size = input_size,
        hidden_size = 64,
        num_layers = 1,
        batch_first = True
    )

    self.to_latent = nn.Linear(
        in_features = 64,
        out_features = 32
    )

    self.from_latent = nn.Linear(
        in_features = 32,
        out_features = 64
    )

    self.decoder = nn.GRU(
        input_size = 64,
        hidden_size = 64,
        num_layers = 1,
        batch_first = True
    )

    self.output = nn.Linear(
        in_features = 64,
        out_features = input_size
    )

  def forward(self, x):
    _, hidden = self.encoder(x)

    hidden = hidden[-1]

    latent = self.to_latent(hidden)
    decoder_hidden = self.from_latent(latent)

    decoder_input = decoder_hidden.unsqueeze(1).repeat(
            1,
            x.size(1),
            1
        )

    decoded, _ = self.decoder(decoder_input)
    output = self.output(decoded)
    return output


In [14]:
epochs = 10
lr = 0.0001
model = GRU_AutoEncoder(4)
model.to(device)
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = 1e-4)

In [15]:
import json
import gc
for epoch in range(epochs):
  total_loss = []
  for batch in train_loader:
    batch = batch.to(device)

    output = model(batch)

    optimizer.zero_grad()

    loss = loss_function(output, batch)

    loss.backward()

    optimizer.step()

    total_loss.append(loss.item())
  print(f"Loss : {np.mean(total_loss)}")
  print("-"*100)

with open("Train_report(1).json", "w") as f:
  json.dump(total_loss, f)
  f.close()

torch.save(model.state_dict(), "gru_autoencoder_model.pth")

del train_loader
del dataset
gc.collect()
torch.cuda.empty_cache()

Loss : 1.1751612476228903
----------------------------------------------------------------------------------------------------
Loss : 1.0415274939272878
----------------------------------------------------------------------------------------------------
Loss : 0.9560192693957495
----------------------------------------------------------------------------------------------------
Loss : 0.8945541148624239
----------------------------------------------------------------------------------------------------
Loss : 0.8512755451719255
----------------------------------------------------------------------------------------------------
Loss : 0.825198343906062
----------------------------------------------------------------------------------------------------
Loss : 0.8081368534817165
----------------------------------------------------------------------------------------------------
Loss : 0.7975818926479409
--------------------------------------------------------------------------------------

# Test - 1

In [16]:
#test - 1

num_cols = ['speed', 'acc', 'change_in_direction', 'is_stationary']
sequence_length = 30

total = []

for i in range(0, 40):

    test_dir = os.path.join(Data, dir[i])
    print(f"Testing in Directory User: {test_dir}")

    trajectory_dir = os.path.join(test_dir, "Trajectory")
    main_data_dir = os.listdir(trajectory_dir)

    for j in range(len(main_data_dir)):

        file_path = os.path.join(
            trajectory_dir,
            main_data_dir[j]
        )

        # Read file
        df = pd.read_csv(
            file_path,
            header=None,
            skiprows=6
        )

        df.columns = [
            "latitude",
            "longitude",
            "zero",
            "altitude",
            "date",
            "datetime",
            "time"
        ]


        df = preprocessing(df)

        if len(df) <= sequence_length:
            continue

        df[num_cols] = scaler.transform(df[num_cols].to_numpy())
        data = df[num_cols].to_numpy(dtype=np.float32)


        n_sequences = len(data) - sequence_length


        sequences = np.lib.stride_tricks.sliding_window_view(
            data,
            window_shape=sequence_length,
            axis=0
        )


        sequences = sequences.transpose(0, 2, 1)

        total.append(sequences)

# Combine all files once
total = np.concatenate(total, axis=0)

dataset = TrajectoryDataset(total)

test_loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=False
)

Testing in Directory User: /content/Geolife Trajectories 1.3/Data/001
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/069
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/012
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/180
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/174
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/064
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/049
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/011
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/061
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/041
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/042
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/072
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/102
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/104
Testing in Directory

In [17]:
model = GRU_AutoEncoder(4)
model.to(device)
criterion = nn.MSELoss()
weights = torch.load("gru_autoencoder_model.pth", map_location=device)
model.load_state_dict(weights)

<All keys matched successfully>

In [18]:
model.eval()
total_loss = []
for batch in test_loader:
  batch = batch.to(device)

  with torch.no_grad():
    output = model(batch)

    loss = loss_function(output, batch)

    total_loss.append(loss.item())

print(f"Loss: {np.mean(total_loss)}")

with open("Test_report(1).json", "w") as f:
  json.dump(total_loss, f)
  f.close()

del test_loader
del dataset
gc.collect()
torch.cuda.empty_cache()

Loss: 0.0922380495839373


# Train - 2

In [21]:
#train - 2

num_cols = ['speed', 'acc', 'change_in_direction', 'is_stationary']
sequence_length = 30

total = []

for i in range(110, len(dir)):

    test_dir = os.path.join(Data, dir[i])
    print(f"Testing in Directory User: {test_dir}")

    trajectory_dir = os.path.join(test_dir, "Trajectory")
    main_data_dir = os.listdir(trajectory_dir)

    for j in range(len(main_data_dir)):

        file_path = os.path.join(
            trajectory_dir,
            main_data_dir[j]
        )

        # Read file
        df = pd.read_csv(
            file_path,
            header=None,
            skiprows=6
        )

        df.columns = [
            "latitude",
            "longitude",
            "zero",
            "altitude",
            "date",
            "datetime",
            "time"
        ]


        df = preprocessing(df)

        if len(df) <= sequence_length:
            continue

        df[num_cols] = scaler.transform(df[num_cols].to_numpy())
        data = df[num_cols].to_numpy(dtype=np.float32)


        n_sequences = len(data) - sequence_length


        sequences = np.lib.stride_tricks.sliding_window_view(
            data,
            window_shape=sequence_length,
            axis=0
        )


        sequences = sequences.transpose(0, 2, 1)

        total.append(sequences)

# Combine all files once
total = np.concatenate(total, axis=0)

dataset = TrajectoryDataset(total)

train_loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=True
)

Testing in Directory User: /content/Geolife Trajectories 1.3/Data/115
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/124
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/086
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/097
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/140
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/177
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/106
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/078
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/119
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/035
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/081
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/170
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/016
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/156
Testing in Directory

In [22]:
epochs = 10
lr = 0.0001
model = GRU_AutoEncoder(4)
model.to(device)
weights = torch.load("gru_autoencoder_model.pth", map_location=device)
model.load_state_dict(weights)
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = 1e-4)


In [23]:
model.train()
for epoch in range(epochs):
  total_loss = []
  for batch in train_loader:
    batch = batch.to(device)

    output = model(batch)

    optimizer.zero_grad()

    loss = loss_function(output, batch)

    loss.backward()

    optimizer.step()

    total_loss.append(loss.item())
  print(f"Loss : {np.mean(total_loss)}")
  print("-"*100)

with open("Train_report(2).json", "w") as f:
  json.dump(total_loss, f)
  f.close()

torch.save(model.state_dict(), "gru_autoencoder_model.pth")

del train_loader
del dataset
gc.collect()
torch.cuda.empty_cache()

Loss : 0.14742403060769022
----------------------------------------------------------------------------------------------------
Loss : 0.14319508586276963
----------------------------------------------------------------------------------------------------
Loss : 0.13939833557772452
----------------------------------------------------------------------------------------------------
Loss : 0.13559983744808796
----------------------------------------------------------------------------------------------------
Loss : 0.1324747567797272
----------------------------------------------------------------------------------------------------
Loss : 0.13006968736398195
----------------------------------------------------------------------------------------------------
Loss : 0.12757115296813717
----------------------------------------------------------------------------------------------------
Loss : 0.12505405826377067
------------------------------------------------------------------------------

# Test - 2

In [32]:
# Test - 2

num_cols = ['speed', 'acc', 'change_in_direction', 'is_stationary']
sequence_length = 30

total = []

for i in range(0,40):

    test_dir = os.path.join(Data, dir[i])
    print(f"Testing in Directory User: {test_dir}")

    trajectory_dir = os.path.join(test_dir, "Trajectory")
    main_data_dir = os.listdir(trajectory_dir)

    for j in range(len(main_data_dir)):

        file_path = os.path.join(
            trajectory_dir,
            main_data_dir[j]
        )

        # Read file
        df = pd.read_csv(
            file_path,
            header=None,
            skiprows=6
        )

        df.columns = [
            "latitude",
            "longitude",
            "zero",
            "altitude",
            "date",
            "datetime",
            "time"
        ]


        df = preprocessing(df)

        if len(df) <= sequence_length:
            continue

        df[num_cols] = scaler.transform(df[num_cols].to_numpy())
        data = df[num_cols].to_numpy(dtype=np.float32)


        n_sequences = len(data) - sequence_length


        sequences = np.lib.stride_tricks.sliding_window_view(
            data,
            window_shape=sequence_length,
            axis=0
        )


        sequences = sequences.transpose(0, 2, 1)

        total.append(sequences)

# Combine all files once
total = np.concatenate(total, axis=0)

dataset = TrajectoryDataset(total)

test_loader = DataLoader(
    dataset,
    batch_size=512,
    shuffle=False
)

Testing in Directory User: /content/Geolife Trajectories 1.3/Data/001
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/069
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/012
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/180
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/174
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/064
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/049
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/011
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/061
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/041
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/042
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/072
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/102
Testing in Directory User: /content/Geolife Trajectories 1.3/Data/104
Testing in Directory

In [33]:
model = GRU_AutoEncoder(4)
model.to(device)
criterion = nn.MSELoss()
weights = torch.load("gru_autoencoder_model.pth", map_location=device)
model.load_state_dict(weights)

<All keys matched successfully>

In [34]:
model.eval()

total_mse = []
for batch in test_loader:
  batch = batch.to(device)

  with torch.no_grad():
    output = model(batch)


    mse = loss_function(output, batch)
    total_mse.append(mse.item())
print(f"mse: {np.mean(total_mse)}")


mse: 0.06977543969308045


In [36]:
threshold = np.percentile(total_mse, 99)
threshold

np.float64(0.3261571645736699)

In [ ]:
del test_loader
del dataset
gc.collect()